<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/CNN_Examples/CNN_Demo4_Inception_and_1x1Conv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 4: GoogLeNet's Inception Module & 1x1 Convolutions

**ELC5365 Deep Learning** | Baylor University

This demo explores two key innovations from GoogLeNet (2014):
1. **1x1 convolutions** — a surprisingly powerful dimensionality reduction tool
2. **Inception Module** — multi-scale feature extraction in a single layer
3. **Global Average Pooling** — replacing massive FC layers

We build a mini-GoogLeNet and train it on CIFAR-10.

**Reference:** Szegedy et al., "Going Deeper with Convolutions," *CVPR*, 2015.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Section 1: The 1x1 Convolution — A Surprising Powerhouse

A **1x1 convolution** seems useless at first — it only looks at one pixel at a time!

But it performs **cross-channel pooling**: combining information across all channels
at each spatial location. This achieves:
1. **Dimensionality reduction** (fewer channels = fewer parameters)
2. **Added non-linearity** (followed by ReLU)

Let's see the dramatic parameter savings.

In [ ]:
# Example: reduce 192 channels to 32 channels at 28x28 spatial resolution
print("=" * 65)
print("Goal: Apply 5x5 conv to go from 192 -> 32 channels at 28x28")
print("=" * 65)

# Approach 1: Direct 5x5 convolution (192 -> 32)
direct = nn.Conv2d(192, 32, kernel_size=5, padding=2)
direct_params = sum(p.numel() for p in direct.parameters())

# Approach 2: 1x1 conv (192 -> 16) then 5x5 conv (16 -> 32)
bottleneck = nn.Sequential(
    nn.Conv2d(192, 16, kernel_size=1),   # 192 -> 16 (reduce)
    nn.ReLU(inplace=True),
    nn.Conv2d(16, 32, kernel_size=5, padding=2)  # 16 -> 32
)
bottleneck_params = sum(p.numel() for p in bottleneck.parameters())

print(f"\nApproach 1 — Direct 5x5 conv:")
print(f"  192 x 32 x 5 x 5 + 32 (bias) = {direct_params:,} parameters")
print(f"\nApproach 2 — 1x1 then 5x5:")
print(f"  1x1: 192 x 16 + 16 = {192*16+16:,}")
print(f"  5x5:  16 x 32 x 5 x 5 + 32 = {16*32*25+32:,}")
print(f"  Total = {bottleneck_params:,} parameters")
print(f"\n  Reduction: {direct_params/bottleneck_params:.1f}x fewer parameters!")

# Verify with actual forward pass
x = torch.randn(1, 192, 28, 28)
y1 = direct(x)
y2 = bottleneck(x)
print(f"\n  Both produce the same output shape: {list(y1.shape)} = {list(y2.shape)}")

---
## Section 2: Implementing the Inception Module

The Inception module applies **4 parallel branches** simultaneously:

| Branch | Operation | Purpose |
|--------|-----------|----------|
| 1 | 1x1 conv | Point-wise features |
| 2 | 1x1 conv -> 3x3 conv | Local features (reduced cost) |
| 3 | 1x1 conv -> 5x5 conv | Wider context (reduced cost) |
| 4 | 3x3 max pool -> 1x1 conv | Pool features |

All outputs are **concatenated** along the channel dimension.

In [ ]:
class InceptionModule(nn.Module):
    """Inception module with 4 parallel branches.
    
    Args:
        in_channels: input channels
        ch1x1: output channels for branch 1 (1x1 conv)
        ch3x3_reduce: channels after 1x1 reduce in branch 2
        ch3x3: output channels for branch 2 (3x3 conv)
        ch5x5_reduce: channels after 1x1 reduce in branch 3
        ch5x5: output channels for branch 3 (5x5 conv)
        pool_proj: output channels for branch 4 (pool projection)
    """
    def __init__(self, in_channels, ch1x1, ch3x3_reduce, ch3x3, ch5x5_reduce, ch5x5, pool_proj):
        super().__init__()
        
        # Branch 1: 1x1 conv
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, 1),
            nn.BatchNorm2d(ch1x1),
            nn.ReLU(inplace=True))
        
        # Branch 2: 1x1 reduce -> 3x3 conv
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3_reduce, 1),
            nn.BatchNorm2d(ch3x3_reduce),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3_reduce, ch3x3, 3, padding=1),
            nn.BatchNorm2d(ch3x3),
            nn.ReLU(inplace=True))
        
        # Branch 3: 1x1 reduce -> 5x5 conv
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5_reduce, 1),
            nn.BatchNorm2d(ch5x5_reduce),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5_reduce, ch5x5, 5, padding=2),
            nn.BatchNorm2d(ch5x5),
            nn.ReLU(inplace=True))
        
        # Branch 4: 3x3 max pool -> 1x1 conv
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU(inplace=True))
    
    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)  # Concatenate channels

# Create an example inception module (matching Inception 3a from GoogLeNet)
inception = InceptionModule(192, ch1x1=64, ch3x3_reduce=96, ch3x3=128,
                            ch5x5_reduce=16, ch5x5=32, pool_proj=32)

x = torch.randn(1, 192, 28, 28)
y = inception(x)
print(f"Input shape:  {list(x.shape)}")
print(f"Output shape: {list(y.shape)}")
print(f"Output channels = 64 + 128 + 32 + 32 = {64+128+32+32} (concatenation of 4 branches)")
print(f"\nParameters: {sum(p.numel() for p in inception.parameters()):,}")

---
## Section 3: Visualizing the Inception Module Branches

In [ ]:
# Show what each branch outputs
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
sample_img, label = cifar10[7]

preprocess = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor()])

# Create a simple feature extractor to get 192-channel features
stem = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
    nn.Conv2d(64, 192, 3, padding=1), nn.ReLU())

x = preprocess(sample_img).unsqueeze(0)
features = stem(x)  # (1, 192, 28, 28)

# Forward through inception and capture each branch
b1 = inception.branch1(features)
b2 = inception.branch2(features)
b3 = inception.branch3(features)
b4 = inception.branch4(features)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Inception Module: What Each Branch "Sees"', fontsize=16, fontweight='bold')

# Original image
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

# Show 2 feature maps from each branch
branches = [('1x1 conv', b1), ('1x1+3x3', b2), ('1x1+5x5', b3), ('pool+1x1', b4)]
for col, (name, feat) in enumerate(branches):
    feat_np = feat[0].detach().numpy()
    axes[0, col+1].imshow(feat_np[0], cmap='viridis')
    axes[0, col+1].set_title(f'{name}\nFilter 0', fontsize=10)
    axes[1, col+1].imshow(feat_np[1], cmap='viridis')
    axes[1, col+1].set_title(f'Filter 1', fontsize=10)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

print("Each branch captures features at different receptive field sizes.")
print("The network learns WHICH scale is most informative for each feature.")

---
## Section 4: Global Average Pooling vs Fully Connected Layers

GoogLeNet replaces the massive FC layers (used in AlexNet/VGG) with
**Global Average Pooling (GAP)**: average each feature map to a single number.

This dramatically reduces parameters.

In [ ]:
# Compare: FC layers vs Global Average Pooling
# Assume 512 feature maps of size 7x7 -> 1000 classes

# Approach 1: Flatten + FC (like VGG)
fc_head = nn.Sequential(
    nn.Flatten(),
    nn.Linear(512 * 7 * 7, 4096),
    nn.Linear(4096, 4096),
    nn.Linear(4096, 1000))
fc_params = sum(p.numel() for p in fc_head.parameters())

# Approach 2: Global Average Pooling + FC (like GoogLeNet)
gap_head = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),  # 512 x 7 x 7 -> 512 x 1 x 1
    nn.Flatten(),
    nn.Linear(512, 1000))
gap_params = sum(p.numel() for p in gap_head.parameters())

print("Feature map: 512 x 7 x 7 -> 1000 classes\n")
print(f"Approach 1 — FC layers (VGG-style):")
print(f"  Parameters: {fc_params:>14,} ({fc_params/1e6:.1f}M)")
print(f"\nApproach 2 — Global Average Pooling (GoogLeNet-style):")
print(f"  Parameters: {gap_params:>14,} ({gap_params/1e6:.1f}M)")
print(f"\n  Reduction: {fc_params/gap_params:.0f}x fewer parameters!")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['FC Layers\n(VGG-style)', 'Global Avg Pool\n(GoogLeNet-style)'],
              [fc_params/1e6, gap_params/1e6],
              color=['#e74c3c', '#2ecc71'], edgecolor='black')
for bar, val in zip(bars, [fc_params/1e6, gap_params/1e6]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{val:.1f}M', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('Parameters (Millions)', fontsize=12)
ax.set_title('Classifier Head Parameter Comparison', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Section 5: Building a Mini-GoogLeNet for CIFAR-10

In [ ]:
class MiniGoogLeNet(nn.Module):
    """Simplified GoogLeNet for CIFAR-10 (32x32 images)."""
    def __init__(self, num_classes=10):
        super().__init__()
        # Stem: simple conv layers
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, 3, padding=1, bias=False),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True))
        
        # Inception modules
        # Output channels: 64+128+32+32 = 256
        self.inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32)
        # Output channels: 128+192+96+64 = 480
        self.inception3b = InceptionModule(256, 128, 128, 192, 32, 96, 64)
        self.pool3 = nn.MaxPool2d(3, stride=2, padding=1)  # 32->16
        
        # Output channels: 192+208+48+64 = 512
        self.inception4a = InceptionModule(480, 192, 96, 208, 16, 48, 64)
        self.pool4 = nn.MaxPool2d(3, stride=2, padding=1)  # 16->8
        
        # Global Average Pooling + Classifier
        self.avgpool = nn.AdaptiveAvgPool2d(1)  # 8x8 -> 1x1
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.pool3(x)
        x = self.inception4a(x)
        x = self.pool4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)

model = MiniGoogLeNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Mini-GoogLeNet: {total_params:,} parameters ({total_params/1e6:.1f}M)")
print(f"For comparison: VGG-16 has 138M parameters")
print(f"\nArchitecture:")
print(f"  Stem -> Inception3a (256ch) -> Inception3b (480ch)")
print(f"  -> Pool -> Inception4a (512ch) -> Pool -> GAP -> FC")

---
## Section 6: Training on CIFAR-10

In [ ]:
# Data loading
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

print(f"Training: {len(trainset):,} images | Test: {len(testset):,} images")

In [ ]:
# Training loop
NUM_EPOCHS = 30  # Increase to 100+ for best results

model = MiniGoogLeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

train_losses, test_accs = [], []
start = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total += inputs.size(0)
    scheduler.step()
    
    train_loss = running_loss / total
    train_acc = 100. * correct / total
    
    # Test
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            correct += model(inputs).argmax(1).eq(targets).sum().item()
            total += inputs.size(0)
    test_acc = 100. * correct / total
    
    train_losses.append(train_loss)
    test_accs.append(test_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
              f"Train: {train_acc:.1f}% | Test: {test_acc:.1f}%")

elapsed = time.time() - start
print(f"\nFinished in {elapsed:.0f}s | Best Test Accuracy: {max(test_accs):.1f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, 'b-', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Mini-GoogLeNet Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(test_accs, 'r-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('Mini-GoogLeNet Test Accuracy', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 7: Architecture Evolution — The Big Picture

| Model | Year | Top-5 Error | Parameters | Key Innovation |
|-------|------|-------------|------------|----------------|
| **AlexNet** | 2012 | 15.3% | 60M | ReLU, Dropout, GPU training |
| **VGGNet** | 2014 | 7.3% | 138M | Small 3x3 filters, depth |
| **GoogLeNet** | 2014 | 6.7% | **5M** | Inception, 1x1 conv, GAP |
| **ResNet** | 2015 | 3.6% | 25M | Skip connections |

GoogLeNet achieved **better accuracy than VGG with 28x fewer parameters!**

The key innovations — 1x1 convolutions, multi-scale processing, and global average
pooling — are now standard building blocks in modern architectures.

In [ ]:
# Compare real architecture parameter counts
archs = {
    'AlexNet': models.alexnet(),
    'VGG-16': models.vgg16(),
    'GoogLeNet': models.googlenet(),
    'ResNet-50': models.resnet50(),
}

errors = {'AlexNet': 15.3, 'VGG-16': 7.3, 'GoogLeNet': 6.7, 'ResNet-50': 3.6}
params = {n: sum(p.numel() for p in m.parameters())/1e6 for n, m in archs.items()}

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71']
for (name, par), err, color in zip(params.items(), errors.values(), colors):
    ax.scatter(par, err, s=300, c=color, edgecolors='black', linewidth=1.5, zorder=5)
    ax.annotate(name, (par, err), textcoords='offset points',
                xytext=(10, 10), fontsize=12, fontweight='bold')

ax.set_xlabel('Parameters (Millions)', fontsize=13)
ax.set_ylabel('ImageNet Top-5 Error (%)', fontsize=13)
ax.set_title('Accuracy vs Model Size', fontsize=15, fontweight='bold')
ax.invert_yaxis()  # Lower error = better
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Lower-left = ideal (few parameters, low error)")
print("GoogLeNet is remarkable: fewest parameters AND low error!")

---
## Summary

| Innovation | Impact |
|------------|--------|
| **1x1 convolution** | ~10x parameter reduction, adds non-linearity |
| **Inception module** | Multi-scale features in a single layer |
| **Global Average Pooling** | Eliminates ~90% of parameters (vs FC layers) |

These ideas are foundational to modern architectures including EfficientNet,
MobileNet, and many others.